In [12]:
# Urban Green Cover Thresholds for PM₂.₅ Mitigation
## Stage 1: Baseline Predictive Modeling (Non-Causal)

# **Objective:**
# Establish baseline predictive performance for ground-level PM₂.₅ across Delhi NCR using multimodal satellite observations, meteorological data, and land-cover context. This notebook evaluates Linear Regression, Random Forest Regressor, and LightGBM Regressor.

# **Methodological Note:** 
# This is purely a predictive baseline. Feature importance extracted in this notebook does NOT imply causality. Causal inference will be explicitly handled in subsequent Double Machine Learning (DML) / Causal Forest stages.

In [13]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error
from sklearn.model_selection import KFold, cross_validate

try:
    import lightgbm as lgb
except ImportError:
    print("CRITICAL: LightGBM is not installed. Please install it using 'pip install lightgbm'.")
    sys.exit(1)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Print Environment Specs
print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"LightGBM version: {lgb.__version__}")

# Plot styling for research quality
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.2)

Python version: 3.13.7
Pandas version: 3.0.5
Numpy version: 2.5.1
LightGBM version: 4.7.0


In [14]:
# Define paths
BASE_DIR = r"C:\Users\Hitakkshi Joshi\Desktop\acm slot 11"
DATA_V2_PATH = os.path.join(BASE_DIR, "data", "ml_ready", "master_modeling_dataset_v2.csv")
TRAIN_PATH = os.path.join(BASE_DIR, "data", "modeling", "splits", "train.csv")
TEST_PATH = os.path.join(BASE_DIR, "data", "modeling", "splits", "test.csv")

RESULTS_DIR = os.path.join(BASE_DIR, "data", "modeling", "results")
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")

# Ensure output directories exist
os.makedirs(PLOTS_DIR, exist_ok=True)

# Load data strictly as IMMUTABLE INPUT
df_v2 = pd.read_csv(DATA_V2_PATH)
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [15]:
# 1. Row counts
assert len(df_v2) == 1615, f"Expected 1615 V2 rows, got {len(df_v2)}"
assert len(df_train) == 1292, f"Expected 1292 train rows, got {len(df_train)}"
assert len(df_test) == 323, f"Expected 323 test rows, got {len(df_test)}"
assert len(df_train) + len(df_test) == len(df_v2), "Train + Test rows do not equal V2 rows"

# 2. Overlap and schemas
train_keys = set(zip(df_train['station'], df_train['year'], df_train['month']))
test_keys = set(zip(df_test['station'], df_test['year'], df_test['month']))
assert len(train_keys.intersection(test_keys)) == 0, "CRITICAL LEAKAGE: Overlap found between train and test station-year-months!"
assert list(df_train.columns) == list(df_test.columns), "Train and Test schemas do not match!"

# 3. Target integrity
assert 'pm25' in df_train.columns and 'pm25' in df_test.columns, "Target 'pm25' is missing!"
assert df_train['pm25'].isnull().sum() == 0, "Missing values in training target!"
assert df_test['pm25'].isnull().sum() == 0, "Missing values in test target!"

# 4. Temporal coverage
assert set(df_train['year'].unique()) == {2022, 2023, 2024, 2025}, "Train missing years!"
assert set(df_test['year'].unique()) == {2022, 2023, 2024, 2025}, "Test missing years!"

# 5. IIT_Delhi condition
assert 'IIT_Delhi' in df_train['station'].values, "IIT_Delhi missing from train!"
assert 'IIT_Delhi' not in df_test['station'].values, "CRITICAL: IIT_Delhi found in test dataset!"

print("All integrity checks passed. V2 Dataset and splits are rigorously validated.")

All integrity checks passed. V2 Dataset and splits are rigorously validated.


In [6]:
TARGET = 'pm25'

# Explicitly exclude metadata, obvious leakage, or categorical identifiers that encourage memorization
cols_to_exclude = ['station', TARGET] 

# Note: We RETAIN year, month, month_sin, month_cos, season_encoded, latitude, and longitude.
# Justification: PM2.5 in Delhi has profound seasonality (winter vs monsoon). Temporal features 
# are scientifically necessary predictors. Lat/Lon provide spatial context for atmospheric patterns.

X_train_raw = df_train.drop(columns=cols_to_exclude)
y_train = df_train[TARGET]

X_test_raw = df_test.drop(columns=cols_to_exclude)
y_test = df_test[TARGET]

# Check for infinites
assert not np.isinf(X_train_raw.select_dtypes(include=np.number)).values.any(), "Infinite values in train!"
assert not np.isinf(X_test_raw.select_dtypes(include=np.number)).values.any(), "Infinite values in test!"

print(f"Predictor count: {X_train_raw.shape[1]}")

Predictor count: 170


In [16]:
# PREPROCESSING PIPELINE (Fitted ONLY on TRAIN)
# Linear models require standardization and imputation (if missing data exists).
# Tree models are robust to unscaled data, but missing data still needs handling for sklearn RF.

# 1. Pipeline for Linear Regression
lr_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

X_train_lr = lr_preprocessor.fit_transform(X_train_raw)
X_test_lr = lr_preprocessor.transform(X_test_raw)

# 2. Pipeline for Random Forest (No scaling needed)
rf_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

X_train_rf = rf_preprocessor.fit_transform(X_train_raw)
X_test_rf = rf_preprocessor.transform(X_test_raw)

# LightGBM handles missing values natively, but we will pass it the raw pandas dataframe
X_train_lgb = X_train_raw.copy()
X_test_lgb = X_test_raw.copy()

print("Preprocessing complete. Scalers/imputers fitted ONLY on training data.")

Preprocessing complete. Scalers/imputers fitted ONLY on training data.


In [17]:
# Define the seed explicitly here just in case
RANDOM_STATE = 42

# Using 5-fold CV on train data only as a diagnostic.
# DISCLAIMER: Standard K-fold on spatial-temporal panel data is inherently optimistic. 
# It does not measure true spatial generalization.

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ('r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error')

# We test RF as the benchmark for CV
cv_results = cross_validate(
    RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    X_train_rf, y_train, cv=kf, scoring=scoring, return_train_score=False
)

cv_metrics = {
    'CV_R2_mean': np.mean(cv_results['test_r2']),
    'CV_R2_std': np.std(cv_results['test_r2']),
    'CV_RMSE_mean': -np.mean(cv_results['test_neg_root_mean_squared_error']),
    'CV_RMSE_std': np.std(cv_results['test_neg_root_mean_squared_error']),
    'CV_MAE_mean': -np.mean(cv_results['test_neg_mean_absolute_error']),
    'CV_MAE_std': np.std(cv_results['test_neg_mean_absolute_error']),
}

print("Cross-validation (Train Only) Complete:")
for k, v in cv_metrics.items():
    print(f"{k}: {v:.4f}")

Cross-validation (Train Only) Complete:
CV_R2_mean: 0.9178
CV_R2_std: 0.0318
CV_RMSE_mean: 16.0917
CV_RMSE_std: 3.3026
CV_MAE_mean: 10.2115
CV_MAE_std: 1.3793


In [18]:
models = {}

# 1. Linear Regression
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train)
models['Linear Regression'] = lr_model

# 2. Random Forest Regressor
print("Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf_model.fit(X_train_rf, y_train)
models['Random Forest'] = rf_model

# 3. LightGBM Regressor
print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_estimators=300, n_jobs=-1)
lgb_model.fit(X_train_lgb, y_train)
models['LightGBM'] = lgb_model

print("All models trained successfully.")

Training Linear Regression...
Training Random Forest...
Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003866 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 33577
[LightGBM] [Info] Number of data points in the train set: 1292, number of used features: 170
[LightGBM] [Info] Start training from score 85.212523
All models trained successfully.


In [19]:
def evaluate_model(y_true, y_pred, model_name, split_name):
    return {
        'model': model_name,
        f'{split_name}_R2': r2_score(y_true, y_pred),
        f'{split_name}_RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        f'{split_name}_MAE': mean_absolute_error(y_true, y_pred),
        f'{split_name}_MedianAE': median_absolute_error(y_true, y_pred)
    }

results = []

for name, model in models.items():
    if name == 'Linear Regression':
        y_train_pred, y_test_pred = model.predict(X_train_lr), model.predict(X_test_lr)
    elif name == 'Random Forest':
        y_train_pred, y_test_pred = model.predict(X_train_rf), model.predict(X_test_rf)
    elif name == 'LightGBM':
        y_train_pred, y_test_pred = model.predict(X_train_lgb), model.predict(X_test_lgb)
    
    train_res = evaluate_model(y_train, y_train_pred, name, 'train')
    test_res = evaluate_model(y_test, y_test_pred, name, 'test')
    
    # Merge
    merged = {**train_res, **test_res, **cv_metrics}
    results.append(merged)

df_results = pd.DataFrame(results)

# Save Baseline Metrics
df_results.to_csv(os.path.join(RESULTS_DIR, 'baseline_model_metrics.csv'), index=False)
display(df_results[['model', 'train_R2', 'test_R2', 'test_RMSE', 'test_MAE']])

,model,train_R2,test_R2,test_RMSE,test_MAE
0,Linear Regression,0.882496,-0.108822,70.230705,51.827836
1,Random Forest,0.989786,-0.166884,72.046009,49.912074
2,LightGBM,0.999453,-0.420690,79.496083,53.762465


In [20]:
# Add Best Model Predictions back to a copy of the test dataframe for diagnostics
df_test_diag = df_test.copy()
df_test_diag['pred_lgb'] = models['LightGBM'].predict(X_test_lgb)
df_test_diag['pred_rf'] = models['Random Forest'].predict(X_test_rf)
df_test_diag['pred_lr'] = models['Linear Regression'].predict(X_test_lr)

# Year-wise
year_metrics = []
for year in sorted(df_test_diag['year'].unique()):
    sub = df_test_diag[df_test_diag['year'] == year]
    year_metrics.append({
        'Year': year,
        'N_obs': len(sub),
        'LGB_R2': r2_score(sub['pm25'], sub['pred_lgb']),
        'LGB_RMSE': np.sqrt(mean_squared_error(sub['pm25'], sub['pred_lgb'])),
        'LGB_MAE': mean_absolute_error(sub['pm25'], sub['pred_lgb'])
    })
df_yearly = pd.DataFrame(year_metrics)
df_yearly.to_csv(os.path.join(RESULTS_DIR, 'yearly_model_metrics.csv'), index=False)

# Season-wise (Assuming season_encoded maps: 1:Winter, 2:Summer, 3:Monsoon, 4:Post-Monsoon)
season_map = {1: 'Winter', 2: 'Summer', 3: 'Monsoon', 4: 'Post-monsoon'}
season_metrics = []
for s_code, s_name in season_map.items():
    if 'season_encoded' in df_test_diag.columns:
        sub = df_test_diag[df_test_diag['season_encoded'] == s_code]
        if len(sub) > 0:
            season_metrics.append({
                'Season': s_name,
                'N_obs': len(sub),
                'LGB_R2': r2_score(sub['pm25'], sub['pred_lgb']),
                'LGB_RMSE': np.sqrt(mean_squared_error(sub['pm25'], sub['pred_lgb'])),
                'LGB_MAE': mean_absolute_error(sub['pm25'], sub['pred_lgb'])
            })
pd.DataFrame(season_metrics).to_csv(os.path.join(RESULTS_DIR, 'seasonal_model_metrics.csv'), index=False)

In [21]:
df_test_diag['residual_lgb'] = df_test_diag['pm25'] - df_test_diag['pred_lgb']
df_test_diag['residual_rf'] = df_test_diag['pm25'] - df_test_diag['pred_rf']
df_test_diag['residual_lr'] = df_test_diag['pm25'] - df_test_diag['pred_lr']

res_summary = {
    'Mean_Residual_LGB': df_test_diag['residual_lgb'].mean(),
    'Median_Residual_LGB': df_test_diag['residual_lgb'].median(),
    'Std_Residual_LGB': df_test_diag['residual_lgb'].std(),
    'Worst_Positive_LGB': df_test_diag['residual_lgb'].max(), # Model severely underpredicted
    'Worst_Negative_LGB': df_test_diag['residual_lgb'].min()  # Model severely overpredicted
}
pd.DataFrame([res_summary]).to_csv(os.path.join(RESULTS_DIR, 'residual_summary.csv'), index=False)

In [22]:
# LightGBM importance (Split / Gain) -> We use default which is number of splits, but Gain is better for scientific interpretation.
lgb_importance = models['LightGBM'].booster_.feature_importance(importance_type='gain')
rf_importance = models['Random Forest'].feature_importances_

feat_imp_df = pd.DataFrame({
    'Feature': X_train_raw.columns,
    'RF_Importance': rf_importance,
    'LGB_Gain': lgb_importance
})
feat_imp_df = feat_imp_df.sort_values(by='LGB_Gain', ascending=False)
feat_imp_df.to_csv(os.path.join(RESULTS_DIR, 'feature_importance.csv'), index=False)

In [23]:
# VISUALIZATION 1: Model Performance
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
models_list = df_results['model']
ax[0].bar(models_list, df_results['test_R2'], color=['#4C72B0', '#55A868', '#C44E52'])
ax[0].set_title('Test R² Score (Higher is better)')
ax[0].set_ylim(0, 1)

ax[1].bar(models_list, df_results['test_RMSE'], color=['#4C72B0', '#55A868', '#C44E52'])
ax[1].set_title('Test RMSE (Lower is better)')

ax[2].bar(models_list, df_results['test_MAE'], color=['#4C72B0', '#55A868', '#C44E52'])
ax[2].set_title('Test MAE (Lower is better)')

plt.suptitle('Baseline Model Comparison on Test Dataset', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '01_model_performance.png'), dpi=300)
plt.close()

# VISUALIZATION 2: Observed vs Predicted
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for i, model_name in enumerate(['Linear Regression', 'Random Forest', 'LightGBM']):
    pred_col = f'pred_{"lr" if i==0 else "rf" if i==1 else "lgb"}'
    axes[i].scatter(df_test_diag['pm25'], df_test_diag[pred_col], alpha=0.5, edgecolor='k')
    axes[i].plot([0, 500], [0, 500], 'r--', lw=2) # 1:1 line
    axes[i].set_title(f'{model_name}\nR²={df_results.loc[i,"test_R2"]:.3f}, RMSE={df_results.loc[i,"test_RMSE"]:.1f}')
    axes[i].set_xlabel('Observed PM₂.₅ (µg/m³)')
    if i == 0: axes[i].set_ylabel('Predicted PM₂.₅ (µg/m³)')
    axes[i].set_xlim(0, 500)
    axes[i].set_ylim(0, 500)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '02_observed_vs_predicted.png'), dpi=300)
plt.close()

# VISUALIZATION 3: Residual Diagnostics
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for i, model_name in enumerate(['Linear Regression', 'Random Forest', 'LightGBM']):
    pred_col = f'pred_{"lr" if i==0 else "rf" if i==1 else "lgb"}'
    res_col = f'residual_{"lr" if i==0 else "rf" if i==1 else "lgb"}'
    axes[i].scatter(df_test_diag[pred_col], df_test_diag[res_col], alpha=0.4, color='purple')
    axes[i].axhline(0, color='red', linestyle='--', lw=2)
    axes[i].set_title(f'{model_name} Residuals')
    axes[i].set_xlabel('Predicted PM₂.₅ (µg/m³)')
    if i == 0: axes[i].set_ylabel('Residual (Observed - Predicted)')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '03_residual_diagnostics.png'), dpi=300)
plt.close()

# VISUALIZATION 4: Spatial Error Map (LightGBM)
station_errors = df_test_diag.groupby('station').agg({
    'latitude': 'mean',
    'longitude': 'mean',
    'residual_lgb': lambda x: np.mean(np.abs(x)) # Mean Absolute Error per station
}).reset_index().rename(columns={'residual_lgb': 'MAE'})

plt.figure(figsize=(10, 8))
scatter = plt.scatter(station_errors['longitude'], station_errors['latitude'], 
                      c=station_errors['MAE'], cmap='YlOrRd', s=100, edgecolor='k')
plt.colorbar(scatter, label='Mean Absolute Test Error (µg/m³)')
plt.title('Spatial Distribution of Model Errors (LightGBM MAE)\n(Note: Descriptive visualization, not causal proof)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
for i, row in station_errors.iterrows():
    plt.annotate(row['station'][:5], (row['longitude'], row['latitude']), fontsize=8, alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '04_spatial_error_map.png'), dpi=300)
plt.close()

# VISUALIZATION 5: Feature Importance
top_n = 20
lgb_top = feat_imp_df.head(top_n)
plt.figure(figsize=(10, 8))
sns.barplot(x='LGB_Gain', y='Feature', data=lgb_top, palette='viridis')
plt.title(f'Top {top_n} Predictors: LightGBM (Gain)\n(Predictive Influence Only, Not Causal Effect)')
plt.xlabel('Feature Gain')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '05_feature_importance.png'), dpi=300)
plt.close()

# VISUALIZATION 6: Environmental Relationship (PM2.5 vs NDVI)
# Using NDVI (assuming column 'NDVI' exists, if named differently, dynamically pick the strongest vegetation feature)
veg_cols = [c for c in df_train.columns if 'ndvi' in c.lower() or 'evi' in c.lower()]
if veg_cols:
    chosen_veg = veg_cols[0]
    plt.figure(figsize=(10, 6))
    sns.regplot(x=df_train[chosen_veg], y=df_train['pm25'], scatter_kws={'alpha':0.2}, line_kws={'color':'red'})
    plt.title(f'Observed PM₂.₅ vs {chosen_veg} (Training Data)\nExploratory Relationship')
    plt.xlabel(chosen_veg)
    plt.ylabel('Observed PM₂.₅ (µg/m³)')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '06_environmental_relationship.png'), dpi=300)
    plt.close()

C:\Users\Hitakkshi Joshi\AppData\Local\Temp\ipykernel_443972\3765132785.py:30: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\Hitakkshi Joshi\AppData\Local\Temp\ipykernel_443972\3765132785.py:30: UserWarning: Glyph 8325 (\N{SUBSCRIPT FIVE}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\Hitakkshi Joshi\AppData\Local\Temp\ipykernel_443972\3765132785.py:31: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig(os.path.join(PLOTS_DIR, '02_observed_vs_predicted.png'), dpi=300)
C:\Users\Hitakkshi Joshi\AppData\Local\Temp\ipykernel_443972\3765132785.py:31: UserWarning: Glyph 8325 (\N{SUBSCRIPT FIVE}) missing from font(s) Arial.
  plt.savefig(os.path.join(PLOTS_DIR, '02_observed_vs_predicted.png'), dpi=300)
C:\Users\Hitakkshi Joshi\AppData\Local\Temp\ipykernel_443972\3765132785.py:44: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\Hitakkshi J

In [ ]:
findings = []
findings.append("=== AUTOMATED RESEARCH DIAGNOSTICS ===")

# Model comparison
lr_r2 = df_results[df_results['model']=='Linear Regression']['test_R2'].values[0]
lgb_r2 = df_results[df_results['model']=='LightGBM']['test_R2'].values[0]
rf_r2 = df_results[df_results['model']=='Random Forest']['test_R2'].values[0]

if lgb_r2 - lr_r2 > 0.1:
    findings.append("- LightGBM substantially outperforms Linear Regression, suggesting nonlinear relationships are highly predictive.")

if rf_r2 - lr_r2 > 0.1:
    findings.append("- Random Forest substantially outperforms Linear Regression.")

# Overfitting check
lgb_train_r2 = df_results[df_results['model']=='LightGBM']['train_R2'].values[0]
if (lgb_train_r2 - lgb_r2) > 0.15:
    findings.append("- Potential Overfitting flagged: LightGBM train R² is notably higher than test R².")

# Error metrics disparity
lgb_rmse = df_results[df_results['model']=='LightGBM']['test_RMSE'].values[0]
lgb_mae = df_results[df_results['model']=='LightGBM']['test_MAE'].values[0]
if lgb_rmse > (lgb_mae * 1.5):
    findings.append("- RMSE is substantially larger than MAE. This indicates the model struggles with extreme PM₂.₅ outlier events.")

# Feature dominance
top_5_features = feat_imp_df['Feature'].head(5).tolist()
if any('ndvi' in f.lower() for f in top_5_features):
    findings.append("- Vegetation features appear highly influential in the predictive model.")
if any('road' in f.lower() or 'pop' in f.lower() for f in top_5_features):
    findings.append("- Urban-context features (population/OSM roads) contribute meaningfully to prediction.")
if not any('worldcover' in f.lower() for f in top_5_features):
    findings.append("- WorldCover 2021 static features show negligible predictive importance relative to dynamic meteorology/seasonality.")

# Outlier check
q95 = df_test_diag['pm25'].quantile(0.95)
findings.append(f"- 95th Percentile PM2.5 in Test: {q95:.2f}. Extreme observations disproportionately impact RMSE.")

with open(os.path.join(RESULTS_DIR, 'findings_report.txt'), 'w') as f:
    f.write('\n'.join(findings))
print("Automated diagnostics generated and saved.")